# Evaluation

Notebook for Lightning evaluation prototyping.

In [1]:
import numpy as np
import pandas as pd
from glob import glob
import torch
from pytorch_lightning import Trainer


from src.model import PoolingMLP, TransformerEncoder

In [2]:
#MODEL_CHKPT = '/home/tliu/learning-ivs/workdir/pool_mlp_lennon_bs256_lr0.001_eps100_hidden256/ckpts/exp_name=pool_mlp_lennon_bs256_lr0.001_eps100_hidden256-val_loss=0.0167.ckpt'
TRANSFORMER_MODEL_CHKPT = '/home/tliu/learning-ivs/checkpoints/linear/transformer_linear_normal_bs2048_lr0.1_eps100/ckpts/exp_name=transformer_linear_normal_bs2048_lr0.1_eps100-val_loss=8.2571-v1.ckpt'


# trainer = Trainer()
# trainer.test(PoolingMLP(input_channels=3, 
#                                 hidden_channels=64, 
#                                 num_classes=1, 
#                                 depth=3), ckpt_path=MODEL_CHKPT)

In [11]:
# model = PoolingMLP.load_from_checkpoint(MODEL_CHKPT)
model = TransformerEncoder.load_from_checkpoint(TRANSFORMER_MODEL_CHKPT, n_blocks=3, n_heads=1, d_model=1, d_hidden=10)

In [8]:
model.eval()

TransformerEncoder(
  (encoder): EncoderBlock(
    (attn_norm): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
    (mh_attn): MultiHeadAttentionBlock(
      (W_q): Linear(in_features=1, out_features=1, bias=True)
      (W_k): Linear(in_features=1, out_features=1, bias=True)
      (W_v): Linear(in_features=1, out_features=1, bias=True)
      (proj): Linear(in_features=1, out_features=1, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (mlp_norm): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
    (mlp): Sequential(
      (0): Linear(in_features=1, out_features=10, bias=True)
      (1): ReLU()
      (2): Linear(in_features=10, out_features=1, bias=True)
    )
    (mlp_dropout): Dropout(p=0.1, inplace=False)
  )
  (model): Sequential(
    (0): EncoderBlock(
      (attn_norm): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
      (mh_attn): MultiHeadAttentionBlock(
        (W_q): Linear(in_features=1, out_fe

In [9]:
data_dir = "/home/tliu/learning-ivs/tmp_lennon100_tau1/test"
preds = []
for file_name in glob(f"{data_dir}/*.parquet"):
    df = pd.read_parquet(file_name)
    data = torch.tensor(df.to_numpy(), dtype=torch.float32)
    #print(data.shape)
    # create a batch dimension
    data = data.unsqueeze(0)
    preds.append(model(data).squeeze().item())
    #break

RuntimeError: Given normalized_shape=[1], expected input with shape [*, 1], but got input of size[1, 1000, 102]

In [36]:
np.mean(preds), np.std(preds)       

(1.0999535884857177, 0.053064317825937894)